# Introduction

This page

# Prerequsisites

This example requires the following external libraries:
- DEAP: Used for creating the optimisation algorithm
- Requests: For sending API calls to the competition API
- SCOOP: A nice library for multiprocessing

These can be installed by:
> pip install deap requests scoop
or 
> python -m install deap requests scoop

Additionally, the web API requires an API key, which is unique to each user. Because of this, it will need to be manually defined by yourself. If you are unsure of your api key, it can be found [here](https://www.balance-competition.tabletopgames.ai/user_settings)

In [ ]:
import requests
import time
import random

# INSERT YOUR API KEY HERE
api_key = ""

# API Parameters
run_type = "fast"
use_local = True # Set to True if you want to use the local server, False for the remote server

# EA Hyperparameters

tournament_size = 10
blended_alpha = 0.5
mutation_indpb = 0.05

population_size = 30
generations = 10

crossover_prob = 0.7
mutation_prob = 0.1

# Constant Values
url_base  = "http://balance-competition.tabletopgames.ai/api/"

# Random object
rand = random.Random()
rand.seed(1337)

# Fitness Function

The most critical decision to make is what to optimise for. In our case we are trying to maximise the balance score returned from the API. So we can defined the fitness function in DEAP like so:

In [ ]:
from deap import creator, base

creator.create("FitnessMax", base.Fitness, weights=(1.0,))

What we are doing here is letting DEAP know that we are trying to maximise a single value.

# Individual Representation

One of the first decisions to make with an evolutionary algorithm is how to represent the problem. The API expects the parameters as a json object in the form of:

> 'param_name': 5

Where 5 is the chosen value. The options for each parameter are a set of discrete integers (expect for two we get to later). 

One option of representing this search space is having each individual comprise of a list of integers, where each integer is the index for a parameter. 

For example for exploding kittens we could have a search space of:

```json
{
    "nCardsPerPlayer": [3, 5, 7, 10, 15],
    "ATTACK_count": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "nopeOwnCards": [true, false]
}
```

An individual would then take the form of:

> [0, 4, 1]

Which would translate to:

```json
{
    "nCardsPerPlayer": 3,
    "ATTACK_count": 5,
    "nopeOwnCards": false
}
```

Because each parameter has a defined set of values it can take, this creates constraints for the optimisation process. To account for this, we will make a penalty function, which penalises each value which corresponds to an invalid index.

A json with all the valid paramters is included with this notebook. Let's load it up now

In [ ]:
import json

with open("valid_params.json", "r") as f:
    valid_params = json.load(f)

## The Terrible Two

Earlier it was mentioned there were two exceptions from the rules. These two parameters are:

### CARDS
**CARDS** is a parameter from Dominion which controls which kingdom cards are brought into the game. It is represented as a list of strings, where each string corresponds to a card name. It has two constraints: One, each card may only be included once. Two, it must contain exactly 10 strings.

### wonders
*wonders* is a parameter from 7 Wonders which controls which wonder boards players can pick. It is also represented as a list of string, which each string is the name of a wonder, Each wonder may only appear once, and there must be 4 - 7 wonders in the list.

## Creating an individual

Based on this, we will reserve 10 integers for *CARDS*, where the value of the integer will be the index of the cards to choose. And we will reserve 7 integers for **wonders**, where again each integer value will be the index of a wonder string. To start with we will find out how many parameters there are, and reserve specific indexes for them.



In [ ]:
from deap import tools

# Constants for the number of cards and wonders
NO_CARDS = 10
MIN_WONDERS = 4
MAX_WONDERS = 7

# The maximum index for wonders and cards
MAX_WONDERS_IDX = 7 # +1 incase they do not want to use a wonder
MAX_CARDS_IDX = 24


# Flatten the valid_params dictionary into a list of valid values (excluding CARDS and individual_size)
params = []
for game in valid_params:
    for param in valid_params[game]:
        if param != "CARDS" and param != "wonders":
            params.append(valid_params[game][param])

# We need to work out the size of the individual based on the parameteers
num_normal_params = len(params)
individual_size = num_normal_params + (NO_CARDS) + (MAX_WONDERS)

# -1 to account for 0 indexing
maximum_indexs = [len(param)-1 for param in params] 

# Helpful variables for slicing the individual
# into normal parameters, cards, and wonders
params_start_idx = 0
params_end_idx = num_normal_params -1

cards_start_idx = params_end_idx + 1
cards_end_idx = cards_start_idx + NO_CARDS - 1

wonders_start_idx = cards_end_idx + 1
wonders_end_idx = individual_size - 1

print("Indexs")
print("Params: " + str(params_start_idx) + "-" + str(params_end_idx))
print("Cards: " + str(cards_start_idx) + "-" + str(cards_end_idx))
print("Wonders: " + str(wonders_start_idx) + "-" + str(wonders_end_idx))

print("\nMaximum Value")
print("Params: " + str(max(maximum_indexs)))
print("Cards: " + str(MAX_CARDS_IDX))
print("Wonders: " + str(MAX_WONDERS_IDX))


creator.create("Individual", list, fitness=creator.FitnessMax)

# toolbox = base.Toolbox()
# toolbox.register("integer", rand.randint, 0, max_value)
# toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.integer, n=individual_size)

Now that we have an idea of the structure of an individual, we need to register it to the toolbox. To reduce the number of invalid values, we are going to construct our individual through three functions:

In [ ]:
random_functions = [lambda: rand.randint(0, maximum_indexs[i]) for i in range(num_normal_params)] + \
                    [lambda: rand.randint(0, MAX_CARDS_IDX) for _ in range(NO_CARDS)] + \
                    [lambda: rand.randint(0, MAX_WONDERS_IDX) for _ in range(MAX_WONDERS)]

def test(ind):
    normal_values = [rand.randint(0, maximum_indexs[i]) for i in range(num_normal_params)]
    card_values = random.sample(range(0, MAX_CARDS_IDX + 1), NO_CARDS)  # Sample without replacement
    wonder_values = random.sample(range(0, MAX_WONDERS_IDX + 1), MAX_WONDERS)  # Sample without replacement
    values = normal_values + card_values + wonder_values
                    # [rand.randint(0, MAX_CARDS_IDX) for _ in range(NO_CARDS)] + \
                    # [rand.randint(0, MAX_WONDERS_IDX) for _ in range(MAX_WONDERS)]
    return creator.Individual(values)

toolbox = base.Toolbox()
#toolbox.register("individual", tools.initCycle, creator.Individual, random_functions, n=1)
toolbox.register("individual", test, creator.Individual)

# test_individual = toolbox.individual()
# print("Test individual: " + str(test_individual))

The above snippet registers the make-up of an invidvidual to the DEAP toolbox, we specifiy that it is a list of integers, with a size equal to the number of values required to represent the parameter space. Each value can be in the range 0 - max_value.

## Penalty Function
Now we need to create a penalty function to punish individuals with invalid values. Luckily DEAP provides a simple way to do this, we need to create a function which takes an individual, and returns true if it is valid, and false if its invalid


In [ ]:
def is_valid(individual):
    # Normal parameters
    for i in range(params_start_idx, params_end_idx + 1):
        if individual[i] < 0 or individual[i] > maximum_indexs[i]:
            return False
        
    used_cards = set()
    # Cards
    for i in range(cards_start_idx, cards_end_idx + 1):
        if individual[i] < 0 or individual[i] > MAX_CARDS_IDX:
            return False
        if individual[i] in used_cards:
            return False
        used_cards.add(individual[i])

    # Wonders
    used_wonders = set()
    for i in range(wonders_start_idx, wonders_end_idx + 1):
        if individual[i] < 0 or individual[i] > MAX_WONDERS_IDX:
            return False
        if individual[i] in used_wonders:
            return False
        
        # 7 is the index for no wonder
        if individual[i] != MAX_WONDERS_IDX: # -1 because the index starts at 0
            used_wonders.add(individual[i])
    
    # Make sure the number of wonders is within the allowed range
    if  len(used_wonders) < MIN_WONDERS or len(used_wonders) > MAX_WONDERS:
        return False
    
    return True
    

## Translating

Now we have an individual to operate evolutionary operations on, and a penalty function to validate the individual. However, we now need a way to turn the list of integers into the JSON object the API will understand. To do this, we will create a translation function.

We iterate through every parameter from the json, using an iterator to get its value if its a normal parameter, and special rules for CARDS and wonders.

In [ ]:
def translate_individual(individual):
    """
    Translate the individual into a dictionary of parameters.
    """
    params_dict = {}
    value = iter(individual)

    for game in valid_params:
        params_dict[game] = {}

        for param in valid_params[game]:
            if param == "CARDS":
                params_dict[game][param] = [valid_params[game][param][card] for card in individual[cards_start_idx : cards_end_idx + 1]]
            elif param == "wonders":
                params_dict[game][param] = [valid_params[game][param][wonder] for wonder in individual[wonders_start_idx : wonders_end_idx + 1] if wonder < MAX_WONDERS]
            else:
                params_dict[game][param] = valid_params[game][param][next(value)]
                
    return params_dict
    
# Test to see if this translation works
test_individual = toolbox.individual()
max_values = maximum_indexs + [MAX_CARDS_IDX] * NO_CARDS + [MAX_WONDERS_IDX] * MAX_WONDERS
print("Test individual: " + str(test_individual))
print("Max Indexs     : " + str(max_values))
print("Length: " + str(len(test_individual)))
print("Maximum values: " + str(len(max_values)))
for i in range(len(test_individual)):
    if test_individual[i] > max_values[i]:
        print(f"Error: Index {i} has value {test_individual[i]} which is greater than the maximum value {max_values[i]}")
print("Is valid: " + str(is_valid(test_individual)))
print("Translated: " + str(translate_individual(test_individual)))


# Evaluation

In an earlier section we defined the fitness function as a being a single objective to maximise the score. We now need to evaluate individuals to find their fitness. In our case, we will use the score returned by a TAG run with the chosen parameters.

We are going to implement two seperate evaluation methods, one which uses the web API, and another which uses the locally hosted version. The fitness function will then call the specified function for each game.

## Local API
The evaluation function for the local api is simple, it sends a post request to the endpoint, and retrieves the processed score.

In [ ]:
def local_run(game, params, run_type, timeout=1200000):
    """
    Run the game locally with the given parameters.
    """
    url = "http://localhost:5173/api/run_game"
    
    body = {
        "game": game,
        "params": params,
        "run_type": run_type,
        "timeout": timeout
    }
    
    response = requests.post(url, json=body)
    
    if response.status_code != 200:
        print(f"Error: {response.status_code} - {response.text}")
        return 0
    
    return response.json()['score']

## Web API
The web API is slightly more complex, involving three API endpoints. This is due to needing to query the progress of the run. The evaluation function is created from these three functions.

In [ ]:
def submit_run(game: str, params:dict, run_type:str) -> dict:
    url = url_base + "submit_run/"

    # Lets the server know the request has a JSON body
    headers = {
        "Content-Type": "application/json"
    }
    
    # The body of the request, contains the required parameters
    data = {
        "game": game,
        "params": params,
        "api_key": api_key,
        "run_type": run_type
    }
    
    response = requests.post(url, json=data, headers=headers)
    
    return response.json()

def query_run(run_id: str) -> dict:
    url = url = url_base + "query_run/"
    
    # the params to pass in the URL
    params = {
        "id": run_id,
    }

    response = requests.get(url, params=params)
    
    return response.json()

def retrieve_result(run_id: str) -> dict:
    url = url = url_base + "retrieve_result/"
    
    params = {
        "id": run_id,
        "api_key": api_key
    }
    
    response = requests.get(url, params=params)
    
    return response.json()

def web_run(game: str, params: dict, run_type: str) -> float:
    id = submit_run(game, params, run_type)['id']

    while True:
        status = query_run(id)['status']
        if status['status'] == 'complete':
            return retrieve_result(id)['score']
        
        # If the run had an error, give it the worst score (0)
        elif status['status'] == 'error':
            return 0
        
        # If the run is still in progress, wait a bit before checking again
        time.sleep(5)

Now we need to create an evluation function which will do either of these methods. 

In [ ]:
def evaluate(individual):

    """
    Evaluate the individual by running the game with the given parameters.
    The individual is a list of integers representing the parameters.
    """
    # Translate the individual into a dictionary of parameters
    params_dict = translate_individual(individual)
    score = 0

    # Each game must be run separately
    for game in params_dict:
        if use_local:
            score += local_run(game, params_dict[game], run_type)
        else:
            score += web_run(game, params_dict[game], run_type)
    
    score /= 4  # Average the score across all games
    return (score,)


Now we need to define the operators we will use for the evolutionary optimisation process. Luckily DEAP comes with a bunch out of the box.

In [ ]:
toolbox.register("population", tools.initRepeat, list, toolbox.individual, n=population_size)
toolbox.register("evaluate", evaluate)
toolbox.decorate("evaluate", tools.DeltaPenalty(is_valid, 0.0))  # Penalize invalid individuals
toolbox.register("select", tools.selTournament, tournsize=tournament_size)
toolbox.register("mate", tools.cxPartialyMatched)
toolbox.register("mutate", tools.mutShuffleIndexes, indpb=mutation_indpb)

With the operators defined, we can now make the evolutionary algorithm loop

In [ ]:
import numpy

stats = tools.Statistics(key=lambda ind: ind.fitness.values)
logbook = tools.Logbook()
stats.register("avg", numpy.mean)
stats.register("std", numpy.std)
stats.register("min", numpy.min)
stats.register("max", numpy.max)


def optimise(population):

    for gen in range(generations):
        print(f"Generation {gen + 1}/{generations}")

        # Select the next generation individuals
        offspring = toolbox.select(population, len(population))
        offspring = list(toolbox.map(toolbox.clone, offspring))

        # Apply crossover and mutation on the offspring
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if rand.random() < crossover_prob:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values

        for mutant in offspring:
            if rand.random() < mutation_prob:
                toolbox.mutate(mutant)
                del mutant.fitness.values

        # Evaluate the individuals with an invalid fitness
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = list(toolbox.map(toolbox.evaluate, invalid_ind))
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit

        # Replace the old population by the offspring
        population[:] = offspring
        record = stats.compile(population)
        logbook.record(gen=gen, **record)
        print(record)

In [ ]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as executor:
    toolbox.register("map", executor.map)
    pop = toolbox.population(n=population_size)
    optimise(pop)

    print("Final Statistics:")
    print(logbook.stream)
